In [58]:
#import the necessary packages
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import pickle

In [59]:
#Load the csv file and conver it into a dataframe
data=pd.read_csv("diabetes.csv")
data.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [60]:
data.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

In [61]:
#Divide the dataset dependent and independent features
X=data.drop('Outcome',axis=True)
y=data['Outcome']

In [62]:
X

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


In [63]:
y

0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: Outcome, Length: 768, dtype: int64

In [64]:
#Split the data into training and testing sets
X_train,X_test,Y_train,Y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [65]:
#Scale these features
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [66]:
with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

ANN IMPLEMENTATION

In [67]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [68]:
#Build our ANN model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32,activation='relu'),
    Dense(1,activation='sigmoid')
])

In [69]:
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_7 (Dense)             (None, 64)                576       
                                                                 
 dense_8 (Dense)             (None, 32)                2080      
                                                                 
 dense_9 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2689 (10.50 KB)
Trainable params: 2689 (10.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [70]:
#Compile the model
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])



In [71]:
#Set up the Tensorboard
import datetime
log_dir="log/fit/"+datetime.datetime.now().strftime("%Y%M%d-%H%M%S")
tensorboard_callback=tf.keras.callbacks.TensorBoard(log_dir=log_dir,histogram_freq=1)

In [72]:
#Setup early stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [75]:
#Train the Model

history=model.fit(
    X_train,Y_train,validation_data=(X_test,Y_test),epochs=100,
    callbacks=[tensorboard_callback,early_stopping_callback],
)

Epoch 1/100
20/20 [==============================] - 0s 18ms/step - loss: 0.5559 - accuracy: 0.7248 - val_loss: 0.6471 - val_accuracy: 0.6429
Epoch 2/100
20/20 [==============================] - 0s 20ms/step - loss: 0.5592 - accuracy: 0.7101 - val_loss: 0.6529 - val_accuracy: 0.6429
Epoch 3/100
20/20 [==============================] - 0s 24ms/step - loss: 0.5501 - accuracy: 0.7296 - val_loss: 0.6536 - val_accuracy: 0.6429
Epoch 4/100
20/20 [==============================] - 0s 20ms/step - loss: 0.5546 - accuracy: 0.7036 - val_loss: 0.6599 - val_accuracy: 0.6429
Epoch 5/100
20/20 [==============================] - 0s 16ms/step - loss: 0.5664 - accuracy: 0.7020 - val_loss: 0.6603 - val_accuracy: 0.6429
Epoch 6/100
20/20 [==============================] - 0s 24ms/step - loss: 0.5579 - accuracy: 0.7166 - val_loss: 0.6630 - val_accuracy: 0.6429
Epoch 7/100
20/20 [==============================] - 0s 22ms/step - loss: 0.5469 - accuracy: 0.7280 - val_loss: 0.6709 - val_accuracy: 0.6429
Epoch 

In [76]:
model.save('model.h5')

c:\Users\AdithyaP\OneDrive\Desktop\DiabetesPrediction\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [77]:
#Load tensorboard extension
%load_ext tensorboard

In [88]:
tensorboard --logdir log/fit

Reusing TensorBoard on port 6007 (pid 1552), started 0:00:05 ago. (Use '!kill 1552' to kill it.)